In [ ]:
# Force-reinstall compatible pandas 2.x and numpy 1.x versions to fix the NamedFunc import issue
!pip install --upgrade "pandas<3.0.0" "numpy<2.0.0"
!pip install quantus tensorflow torch torchvision

import os
import glob
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
import quantus
from PIL import Image
from torchvision import transforms

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 87.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 103.5 MB/s  0:00:00
  Created wheel for numpy: filename=numpy-1.26.4-cp313-cp313-linux_x86_64.whl size=7039806 sha256=ffba983ebffe5bb9cb9907a733ccd9f837b4d986055a362d8cdccf9d514436e8
  Stored in directory: /root/.cache/pip/wheels/8b/2d/9f/b6b46373f328e2ef50388915d351ccacbedac929459b5459bf
Successfully built numpy
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
  Attempting uninstall: pandas
    Found existing installation: pandas 3.1.1
    Uninstalling pandas-3.1.1:
      Successfully uninstalled pandas-3.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]
ERROR: pip's de

  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.


In [2]:

# 1. Unzip the explanations file
print("Unzipping explanations.zip...")
!unzip -q -o /content/explanations.zip -d /content/explanations_extracted

# Update paths based on extraction structure
base_dir = "/content/explanations_extracted"
images_dir = os.path.join(base_dir, "original")
explanations_dir = os.path.join(base_dir, "gradcam")

# 2. Load and preprocess the 100 images and explanations
print("Loading images and explanations...")
image_paths = sorted(glob.glob(os.path.join(images_dir, "*")))[:100]
explanation_paths = sorted(glob.glob(os.path.join(explanations_dir, "*")))[:100]


Unzipping explanations.zip...
Loading images and explanations...


In [10]:
# Image transformations (ResNet-18 standard)
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

x_list = []
saliency_list = []

# Using the active file lists 'image_fps' and 'explanation_fps' loaded in the environment
for img_fp, exp_fp in zip(image_fps, explanation_fps):
    # Load and preprocess image
    img = Image.open(img_fp).convert("RGB")
    x_list.append(transform(img).numpy())

    # Load and preprocess explanation (saliency map)
    if exp_fp.endswith(".npy"):
        exp = np.load(exp_fp)
    else:
        exp = Image.open(exp_fp).convert("L")
        exp = exp.resize((224, 224))
        exp = np.array(exp) / 255.0
    saliency_list.append(exp)

x_batch = np.array(x_list).astype(np.float32)
saliency_batch = np.array(saliency_list).astype(np.float32)

# Assume dummy targets for model predictions
y_batch = np.zeros(len(x_batch), dtype=int)

print(f"Loaded {len(x_batch)} images with shape {x_batch.shape}")
print(f"Loaded {len(saliency_batch)} explanations with shape {saliency_batch.shape}")

# 3. Define a model wrapper for Quantus
model = torchvision.models.resnet18(pretrained=True)
model.eval()

# 4. Evaluate Faithfulness using Pixel Flipping
print("\n--- Evaluating Faithfulness (Pixel Flipping) ---")
pixel_flipping = quantus.PixelFlipping(
    features_in_step=1024,
    perturb_baseline="black",
    perturb_func=quantus.perturb_func.baseline_replacement_by_indices,
    order="descending"
)

# Ensure we match evaluation data limits (up to 100 images)
faithfulness_scores = pixel_flipping(
    model=model,
    x_batch=x_batch[:100],
    y_batch=y_batch[:100],
    saliency_batch=saliency_batch[:100],
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Pixel Flipping Mean Score: {np.mean(faithfulness_scores):.4f}")

# 5. Evaluate Localization using Pointing Game
bbox_batch = []
for _ in range(len(x_batch[:100])):
    # Using a central bounding box coordinates as a fallback metric anchor
    bbox_batch.append([{"box_coords": [50, 50, 170, 170], "label": 0}])

print("\n--- Evaluating Localization (Pointing Game) ---")
pointing_game = quantus.PointingGame()

localization_scores = pointing_game(
    model=model,
    x_batch=x_batch[:100],
    y_batch=y_batch[:100],
    saliency_batch=saliency_batch[:100],
    bboxes=bbox_batch,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Pointing Game Mean Score: {np.mean(localization_scores):.4f}")

NameError: name 'image_paths' is not defined